In [ ]:
#%pip install semopy pandas numpy

import pandas as pd
from semopy import Model, Optimizer



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.9 MB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
  Obtaining dependency information for numdifftools from https://files.pythonhosted.org/packages/a3/5c/37cd5db8c465db2664b2219410b8bc7743da6edb1b616b5d13008bd7cac2/numdifftools-0.9.41-py2.py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 10.6 MB/s eta 0:00:00
  Created wheel for semopy: filename=semopy-2.3.11-py3-none-any.whl size=1659681 sha256=27f4542d818124e6aefbac9424196f813774788a72bd59d88f44a9ab66beab32
  Stored in directory: /Users/zanderholleran/Library/Caches/pip/wheels/d2/9a/31/fae291ff6a649bad125037eef8c7cc63d8c542e14bdcccea37
Successfully built semopy
Note: you may need to restart the kernel to use updated packages.


NameError: name 'df' is not defined

In [5]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

# ---- Latent Real Property Value ----
# Base latent RPV: centered normal
RPV = np.random.normal(0, 1, n)

# ---- Indicators (measurement variables) ----
# BedsBathsSqft ~ lognormal, depends on RPV
BedsBathsSqft = np.exp(7 + 0.5 * RPV + np.random.normal(0, 0.3, n))  # ≈ 700–7000 typical
BedsBathsSqft = np.clip(BedsBathsSqft, 700, 7000)

# Ordinal variables 1–3, correlated with RPV
def ordinal_from_latent(latent):
    # map latent values to 1,2,3 by tertiles
    q = np.quantile(latent, [1/3, 2/3])
    return np.where(latent < q[0], 1,
           np.where(latent < q[1], 2, 3))

Condition_lat = 0.8 * RPV + np.random.normal(0, 1, n)
LotViews_lat = 0.7 * RPV + np.random.normal(0, 1, n)
Location_lat = 0.9 * RPV + np.random.normal(0, 1, n)

Condition = ordinal_from_latent(Condition_lat)
LotViews = ordinal_from_latent(LotViews_lat)
LocationFE = ordinal_from_latent(Location_lat)

# Years_old (inverse of year built) — older homes have slightly lower RPV
YearsOld = np.maximum(5, 80 - 15*RPV + np.random.normal(0, 10, n))

# ---- Transaction-level causes ----
PersonalPropertyValue = np.random.normal(0, 1, n)
LoanTerms = np.random.normal(0, 1, n)
BuyerUrgency = np.random.normal(0, 1, n)
SellerUrgency = np.random.normal(0, 1, n)
SideDeals = np.random.normal(0, 1, n)

# ---- Sale Price (dependent variable) ----
# Combine latent and observed drivers
SalePrice = (
    300000
    + 50000 * RPV
    + 20000 * PersonalPropertyValue
    + 15000 * LoanTerms
    + 10000 * BuyerUrgency
    - 12000 * SellerUrgency
    + 8000 * SideDeals
    + np.random.normal(0, 30000, n)
)

# ---- Package into DataFrame ----
df = pd.DataFrame({
    "SalePrice": SalePrice,
    "BedsBathsSqft": BedsBathsSqft,
    "Condition": Condition,
    "LotViews": LotViews,
    "LocationFE": LocationFE,
    "YearsOld": YearsOld,
    "PersonalPropertyValue": PersonalPropertyValue,
    "LoanTerms": LoanTerms,
    "BuyerUrgency": BuyerUrgency,
    "SellerUrgency": SellerUrgency,
    "SideDeals": SideDeals,
})

df.head()


,SalePrice,BedsBathsSqft,Condition,LotViews,LocationFE,YearsOld,PersonalPropertyValue,LoanTerms,BuyerUrgency,SellerUrgency,SideDeals
0,264043.429747,2139.149732,2,1,2,68.311691,-1.114081,0.785185,-0.033025,0.765402,-0.678495
1,246427.015170,1350.537519,2,1,2,77.539823,-0.630931,-1.777681,-0.503650,1.073413,-0.305499
2,345099.926146,1543.385069,2,2,3,52.328240,-0.942060,0.714746,-0.172375,0.498690,-0.597381
3,415681.898734,1934.170623,3,3,3,53.853650,-0.547996,-0.233724,0.714732,-1.942498,0.110418
4,351376.580451,1202.779379,1,2,1,90.840591,-0.214150,0.707458,1.277857,-0.155422,1.197179


In [ ]:
# model specification
model_desc = """
# Latent measurement model
RealPropertyValue =~ BedsBathsSqft + Condition + LotViews + LocationFE + YearsOld

# Structural part (SalePrice depends on latent and observed determinants)
SalePrice ~ RealPropertyValue + PersonalPropertyValue + LoanTerms + BuyerUrgency + SellerUrgency + SideDeals
"""

# Create the model
model = Model(model_desc)
model.fit(df)

# Summaries
estimates = model.inspect()
estimates



,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,BedsBathsSqft,~,RealPropertyValue,1.000000e+00,-,-,-
1,Condition,~,RealPropertyValue,1.912225e-01,1.504226,0.127124,0.898843
2,LotViews,~,RealPropertyValue,1.557141e-01,1.224922,0.127122,0.898844
3,LocationFE,~,RealPropertyValue,2.045989e-01,1.609444,0.127124,0.898842
4,YearsOld,~,RealPropertyValue,-5.754915e+00,45.269914,-0.127124,0.898842
5,SalePrice,~,RealPropertyValue,4.699440e+01,744.015249,0.063163,0.949637
6,SalePrice,~,PersonalPropertyValue,-6.472371e-02,1411.142692,-0.000046,0.999963
7,SalePrice,~,LoanTerms,-6.578092e-02,1388.822739,-0.000047,0.999962
8,SalePrice,~,BuyerUrgency,-4.250691e-02,1430.802729,-0.00003,0.999976
9,SalePrice,~,SellerUrgency,4.198790e-02,1499.991503,0.000028,0.999978


In [10]:
# Imputed latent scores (factor scores for RealPropertyValue)
rpv_scores = model.predict_factors(df)
df["RPV_hat"] = rpv_scores["RealPropertyValue"]

# Examine correlation between latent scores and SalePrice
print(df[["RPV_hat", "SalePrice"]].corr())


           RPV_hat  SalePrice
RPV_hat    1.00000    0.64407
SalePrice  0.64407    1.00000
